# Morgan fingerprint 2D baseline — Colab

This notebook runs the 2D control for the 3D Uni-Mol LoRA benchmark: Morgan fingerprints (radius 2, 2048 bits) plus per-label balanced logistic regression, on the **identical scaffold folds** with the **identical validation-locked threshold protocol**. It answers one question: *does the 3D foundation model beat a free fingerprint on the same evidence?*

A CPU-only runtime is enough — there is no Uni-Mol featurisation and no training epochs. The full five-fold run takes minutes.

Before running: the repository clone below must contain `scripts/fingerprint_baseline.py`.

In [ ]:
%pip install -q rdkit pandas pyarrow scikit-learn openpyxl

import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# torch is preinstalled on Colab; it is only imported transitively via src.sensory.

## Get the current project

Replace `REPO_URL` if needed. The clone must include the fingerprint baseline script; an older commit will fail the guard below.

In [ ]:
from pathlib import Path
import subprocess, sys

REPO_URL = 'https://github.com/FufanLu/molecular-foundation-model.git'  # replace with your pushed branch/repository
repo_dir = Path('/content/molecular-foundation-model')
if not repo_dir.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only'], check=True)
print('project commit:', subprocess.check_output(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], text=True).strip())
sys.path.insert(0, str(repo_dir))
required_code = [repo_dir / 'src/dataset/sensory.py', repo_dir / 'src/sensory/metrics.py', repo_dir / 'scripts/fingerprint_baseline.py']
missing_code = [str(path) for path in required_code if not path.exists()]
if missing_code:
    raise RuntimeError('This clone is missing the fingerprint baseline implementation. Push/upload the current project first: ' + ', '.join(missing_code))
%cd /content/molecular-foundation-model

## Upload the four raw files

This cell opens Colab's file picker only when a required file is absent. Select all four files together: `leffingwell_merged.csv`, `goodscents_merged.csv`, `flavordb_merged.csv`, and `ChemTastesDB_database.xlsx`.

In [ ]:
from google.colab import files
import shutil

raw_dir = Path('data/raw')
raw_dir.mkdir(parents=True, exist_ok=True)
required_data = {'leffingwell_merged.csv', 'goodscents_merged.csv', 'flavordb_merged.csv', 'ChemTastesDB_database.xlsx'}
missing_data = required_data - {path.name for path in raw_dir.iterdir()}
if missing_data:
    print('Upload:', sorted(missing_data))
    uploaded = files.upload()
    for filename in uploaded:
        if filename in required_data:
            shutil.move(filename, raw_dir / filename)
missing_data = required_data - {path.name for path in raw_dir.iterdir()}
if missing_data:
    raise FileNotFoundError(f'Missing required raw files: {sorted(missing_data)}')
print('Raw data ready:', sorted(required_data))

## Build the sensory-v3 dataset

Identical preparation to the Uni-Mol run: same molecules, same folds, same masked targets. The audit numbers should match the data card (134 exact odor–taste pairs, 100 sour, 47 salty).

In [ ]:
!python -m src.dataset.sensory --raw-dir data/raw --chem-tastes data/raw/ChemTastesDB_database.xlsx --output-dir data/processed/sensory --n-splits 5

import json
audit = json.loads(Path('data/processed/sensory/audit.json').read_text())
print('curated taste labels:', audit['taste_label_counts'])
print('core taste-labelled molecules:', audit['core_taste_labeled_molecule_count'])
print('exact odor–taste pairs:', audit['paired_molecule_count'])
print('low-shot sour/salty examples:', audit['sour_molecule_count'], audit['salty_molecule_count'])

## Run the five-fold fingerprint baseline

Same folds, same masked targets, same validation-locked threshold protocol as the Uni-Mol run. Per-label balanced logistic regression is the 2D analog of the pos_weight-capped BCE. Metrics land in `outputs/v3_fingerprint_lr/` and existing fold files are skipped, so a rerun resumes cheaply.

In [ ]:
!PYTHONPATH=/content/molecular-foundation-model:$PYTHONPATH python scripts/fingerprint_baseline.py --data data/processed/sensory/molecules.parquet --output-dir outputs/v3_fingerprint_lr --folds 0,1,2,3,4

## Aggregate all five folds

Run this only after `fold0_metrics.json` through `fold4_metrics.json` all exist under `outputs/v3_fingerprint_lr/`. Download `reports/v3_fingerprint_lr_5fold/` and the fold metrics, and commit them to the repository for provenance.

In [ ]:
!python scripts/aggregate_cross_sensory.py --metrics outputs/v3_fingerprint_lr/fold*_metrics.json --output-dir reports/v3_fingerprint_lr_5fold

## Side-by-side with the Uni-Mol LoRA five-fold result

The 3D run's aggregate summary ships with the repository (`reports/v3_prototype_d_5fold/summary.json`), so this cell prints the head-to-head directly. Per label, the rightmost column marks which representation wins on mean F1 — expect the answer to differ between common and long-tail labels.

In [ ]:
nn = json.loads(Path('reports/v3_prototype_d_5fold/summary.json').read_text())
fp = json.loads(Path('reports/v3_fingerprint_lr_5fold/summary.json').read_text())

print(f"{'metric':<18} {'Uni-Mol LoRA (3D)':>18} {'Morgan FP + LR (2D)':>20}")
for label, modality in [('odor macro-F1', 'odor'), ('taste macro-F1', 'taste')]:
    a, b = nn['test'][modality]['macro'], fp['test'][modality]['macro']
    print(f"{label:<18} {a['mean']:>9.4f} ± {a['std']:.4f} {b['mean']:>11.4f} ± {b['std']:.4f}")
a, b = nn['test']['score'], fp['test']['score']
print(f"{'combined score':<18} {a['mean']:>9.4f} ± {a['std']:.4f} {b['mean']:>11.4f} ± {b['std']:.4f}")

print('\nper-label mean F1 (3D vs 2D):')
for modality in ('odor', 'taste'):
    for label in sorted(nn['test'][modality]):
        if label == 'macro':
            continue
        a, b = nn['test'][modality][label]['mean'], fp['test'][modality][label]['mean']
        print(f"  {modality}/{label:<15} {a:.3f}  vs  {b:.3f}   {'3D' if a > b else '2D'}")